# CareLoop Clinics — No-Show Guard
## Part 4: Generative AI (Tasks 4.1–4.4)

Run in Google Colab. The Gemini key is entered at runtime with hidden input and must never be committed. The model name is requested because the brief says to use the Flash model still available in AI Studio / Session 22.1. This notebook reuses V2 outputs from Task 4.2 in Task 4.3, so its target API call count is 12.

In [ ]:
!pip -q install google-genai pydantic
from pathlib import Path
from google.colab import files
import pandas as pd, json, time, getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
Path('data').mkdir(exist_ok=True); Path('outputs').mkdir(exist_ok=True)
for filename, target in [('followup_notes.csv',Path('data/followup_notes.csv')),('appointments_clean.csv',Path('outputs/appointments_clean.csv')),('test_risk_scores.csv',Path('outputs/test_risk_scores.csv'))]:
    if not target.exists():
        print('Upload', filename)
        up=files.upload()
        match=next((content for name,content in up.items() if name.lower()==filename.lower()),None)
        if match is None: raise FileNotFoundError(filename)
        target.write_bytes(match)
MODEL_NAME=input('Enter the Gemini Flash model name available in AI Studio / Session 22.1: ').strip()
API_KEY=input('Enter Gemini API key at runtime (do not type it into a code cell): ')
client=genai.Client(api_key=API_KEY)
api_calls=0
ALLOWED={'forgot_or_no_reminder','distance_or_transport','cost_or_fee','illness_or_emergency','work_or_schedule','other'}
SCHEMA={'type':'OBJECT','properties':{'root_cause':{'type':'STRING'},'confidence':{'type':'NUMBER'},'summary':{'type':'STRING'},'recommended_action':{'type':'STRING'}},'required':['root_cause','confidence','summary','recommended_action']}
SYSTEM_TEXT='You are a careful healthcare operations text analyst. Extract only what the note supports. Do not diagnose. Do not invent details. Return valid JSON matching the supplied schema.'

def craft_prompt(note_text):
    return f'''CONTEXT: CareLoop Clinics reviews call-centre notes after missed outpatient appointments.
ROLE: You are a careful healthcare operations analyst.
ACTION: Classify the main reason for the missed appointment and suggest a practical clinic follow-up. Use exactly one root_cause from: {sorted(ALLOWED)}. If the note is unclear, use other; do not guess.
TONE: Neutral, concise, respectful; no diagnosis.
FORMAT: JSON with root_cause, confidence (0 to 1), summary (maximum 25 words), recommended_action.
NOTE: {note_text}'''

def call_gemini(prompt, system_text=SYSTEM_TEXT):
    global api_calls
    # Use minimal thinking when supported by the selected Flash model.
    try:
        config=types.GenerateContentConfig(system_instruction=system_text, temperature=0, max_output_tokens=1000, response_mime_type='application/json', response_schema=SCHEMA, thinking_config=types.ThinkingConfig(thinking_level='minimal'))
    except Exception:
        config=types.GenerateContentConfig(system_instruction=system_text, temperature=0, max_output_tokens=1000, response_mime_type='application/json', response_schema=SCHEMA)
    response=client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config)
    api_calls += 1
    raw=response.text or ''
    try:
        parsed=json.loads(raw)
    except json.JSONDecodeError as exc:
        exc.raw_response=raw
        raise
    return parsed, raw

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def analyse_note(note_text, return_raw=False):
    data, raw = call_gemini(craft_prompt(note_text))
    return (data, raw) if return_raw else data

def validate_insight(data):
    insight=FollowupInsight(**data)
    if insight.root_cause not in ALLOWED:
        raise ValueError(f'root_cause must be one of {sorted(ALLOWED)}')
    return insight
print('Gemini client ready. Do not print or commit the API key.')

## Task 4.1 — One validated call on N001 (5 marks)

In [ ]:
notes=pd.read_csv('data/followup_notes.csv')
note1=notes.loc[notes['note_id']=='N001','note_text'].iloc[0]
data_n001=analyse_note(str(note1))
insight_n001=validate_insight(data_n001)
print(insight_n001.model_dump())
print('Expected N001 root_cause: forgot_or_no_reminder')

## Task 4.2 — Validate and process the first 10 notes (7 marks)

N004 is blank and must be skipped without an API call. Keep invalid raw replies unchanged.

In [ ]:
rows=[]
for _, row in notes.head(10).iterrows():
    note_id=row['note_id']; appointment_id=row['appointment_id']; note_text=row['note_text']
    if pd.isna(note_text) or str(note_text).strip()=='':
        rows.append({'note_id':note_id,'appointment_id':appointment_id,'root_cause':'other','confidence':0.0,'summary':'No note recorded','recommended_action':'Re-call patient','validation_status':'skipped_blank','raw_response':''})
        continue
    time.sleep(13)  # free-tier request spacing
    raw=''; data={}
    try:
        data, raw=analyse_note(str(note_text), return_raw=True)
        validated=validate_insight(data)
        rec=validated.model_dump(); status='valid'
    except Exception as exc:
        raw=getattr(exc,'raw_response',raw)
        rec={k:data.get(k,'') for k in ['root_cause','confidence','summary','recommended_action']} if isinstance(data,dict) else {k:'' for k in ['root_cause','confidence','summary','recommended_action']}
        status='invalid'
        print(f'{note_id} validation/parsing error: {exc}')
    rows.append({'note_id':note_id,'appointment_id':appointment_id,**rec,'validation_status':status,'raw_response':raw})
notes_out=pd.DataFrame(rows,columns=['note_id','appointment_id','root_cause','confidence','summary','recommended_action','validation_status','raw_response'])
notes_out.to_csv('outputs/notes_analysed.csv',index=False)
print(notes_out[['note_id','root_cause','confidence','validation_status']].to_string(index=False))
print('Task 4.2 Gemini calls:',api_calls-1,'(N004 blank must be skipped)')

## Task 4.3 — Break it on purpose (5 marks)

Two deliberate Pydantic failures are caught. V1 is called for N007 and N009. V2 is reused from the CRAFT outputs in Task 4.2.

In [ ]:
print('Invalid reply 1: confidence above 1')
try:
    validate_insight({'root_cause':'other','confidence':1.7,'summary':'A sufficiently long summary','recommended_action':'Call patient again'})
except ValidationError as exc:
    print(exc)
print('Invalid reply 2: missing recommended_action')
try:
    validate_insight({'root_cause':'other','confidence':0.5,'summary':'A sufficiently long summary'})
except ValidationError as exc:
    print(exc)

def simple_prompt(note_text):
    return 'Classify this note: ' + str(note_text)

comparison=[]
notes_by_id=notes.set_index('note_id')
notes_out_by_id=notes_out.set_index('note_id')
for nid in ['N007','N009']:
    raw_note=notes_by_id.loc[nid,'note_text']
    time.sleep(13)
    try:
        v1_data,v1_raw=call_gemini(simple_prompt(raw_note))
        v1_label=v1_data.get('root_cause','')
        v1_valid=v1_label in ALLOWED
    except Exception as exc:
        v1_label=''; v1_raw=''; v1_valid=False; print('V1 error',nid,exc)
    # V2 is reused from the CRAFT result in Task 4.2, avoiding duplicate API calls.
    v2_label=notes_out_by_id.loc[nid,'root_cause']
    v2_valid=(v2_label in ALLOWED) and (notes_out_by_id.loc[nid,'validation_status']=='valid')
    comparison.extend([
        {'note_id':nid,'prompt':'V1: Classify this note','root_cause':v1_label,'allowed_category':v1_valid,'raw_response':v1_raw},
        {'note_id':nid,'prompt':'V2: CRAFT + allowed categories','root_cause':v2_label,'allowed_category':v2_valid,'raw_response':notes_out_by_id.loc[nid,'raw_response']}
    ])
comparison_df=pd.DataFrame(comparison)
print(comparison_df[['note_id','prompt','root_cause','allowed_category']].to_string(index=False))
print('Total Gemini API calls actually made across Tasks 4.1–4.3:',api_calls)

## Task 4.4 — Merge reasons and produce the recommendation evidence (5 marks)

In [ ]:
appointments=pd.read_csv('outputs/appointments_clean.csv')
risk=pd.read_csv('outputs/test_risk_scores.csv')
merged=notes_out.merge(appointments[['appointment_id','reminder_sent','status','no_show']],on='appointment_id',how='left')
merged=merged.merge(risk[['appointment_id','risk_prob','flag']],on='appointment_id',how='left')
print('Merged note rows:',len(merged))
print('Root cause by reminder status:')
print(pd.crosstab(merged['root_cause'],merged['reminder_sent'],dropna=False))
root_counts=merged['root_cause'].value_counts(dropna=False)
print('Root-cause counts:')
print(root_counts)
print('Risk-score output rows:',len(risk),'test rows flagged:',int(risk['flag'].sum()))
top_reason=str(root_counts.index[0]) if len(root_counts) else 'other'
recommendation=(f"CareLoop should prioritise next-day calls for appointments flagged by the selected Logistic Regression model at the 0.35 threshold; the held-out test set achieved 0.571 recall and 0.789 ROC-AUC. Historical analysis shows the 31+ day booking band had a 60.5% no-show rate versus 16.2% for 1–7 days, so long-lead bookings deserve earlier confirmation. Hadapsar Care Centre recorded a 27.3% no-show rate among 143 appointments. In the ten-note illustration, the most frequent extracted reason was {top_reason}; use the corresponding suggested action for individual follow-up, but do not generalise this small sample to all clinics.")
print('\nREADME recommendation paragraph — copy after checking the actual AI outputs:')
print(recommendation)
print('Limitations to state: ten notes are illustrative; imputation/outlier statistics were fitted before the split; random split/patient overlap/observational data limit causal and future-time conclusions.')